<a href="https://colab.research.google.com/github/karthikoo7/Machine_Learning-BDA-/blob/main/KernelPCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd


In [2]:
df = pd.read_csv('/content/ev_charging_patterns.csv')


In [3]:
df.isna().sum()

,0
User ID,0
Vehicle Model,0
Battery Capacity (kWh),0
Charging Station ID,0
Charging Station Location,0
Charging Start Time,0
Charging End Time,0
Energy Consumed (kWh),66
Charging Duration (hours),0
Charging Rate (kW),66


In [4]:
df.dtypes

,0
User ID,object
Vehicle Model,object
Battery Capacity (kWh),float64
Charging Station ID,object
Charging Station Location,object
Charging Start Time,object
Charging End Time,object
Energy Consumed (kWh),float64
Charging Duration (hours),float64
Charging Rate (kW),float64


In [5]:
df.nunique()

,0
User ID,1320
Vehicle Model,5
Battery Capacity (kWh),147
Charging Station ID,462
Charging Station Location,5
Charging Start Time,1320
Charging End Time,1309
Energy Consumed (kWh),1254
Charging Duration (hours),1320
Charging Rate (kW),1254


In [6]:
df.drop(columns=['User ID','Charging Station ID','Charging Start Time','Charging End Time'],inplace=True)
df.shape

(1320, 16)

In [7]:
df.drop_duplicates(inplace=True)
df.shape

(1320, 16)

In [8]:
df.dropna(inplace=True)
df.shape

(1131, 16)

In [9]:
df_ohe = pd.get_dummies(df)
df_ohe.shape

(1131, 37)

Kernel PCA

In [10]:
X = df_ohe

In [11]:
from sklearn.model_selection import train_test_split
X_train,X_test = train_test_split(X,test_size=0.3,random_state=7)

X_train.shape,X_test.shape

((791, 37), (340, 37))

In [12]:
X_train = (X_train - X_train.mean())/X_train.std() # standard scale formula internally -- directly procedure vs using standardscaler()
X_test = (X_test - X_train.mean())/X_train.std()

In [13]:
# poly,RBF,cosine
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(kernel='poly',degree=2,random_state=7)
kpca.fit(X_train)

KernelPCA(degree=2, kernel='poly', random_state=7)

In [14]:
kpca.eigenvalues_.shape

(474,)

In [15]:
# poly,RBF,cosine
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(kernel='rbf',random_state=7)
kpca.fit(X_train)

kpca.eigenvalues_.shape

(790,)

In [16]:
## just seeing if imputing outliers and running kernel pca on it , makes a change
def outlier_impute_iqr(df,col):
  df1 =  df.copy()
  q1,q3 = df1[col].quantile([0.25,0.75])
  iqr = q3-q1
  lb = q1 - 1.5 * iqr
  ub = q3 + 1.5 * iqr
  print("Column-->",col)
  df1.loc[(df1[col]<lb),col] = lb
  df1.loc[(df1[col]>ub),col] = ub
  print(lb,ub,df1[col].min(),df1[col].max())
  return df1

In [17]:
# Separating out continous columns

con_col = X_train.columns[X_train.nunique() > 15]
con_col

Index(['Battery Capacity (kWh)', 'Energy Consumed (kWh)',
       'Charging Duration (hours)', 'Charging Rate (kW)',
       'Charging Cost (USD)', 'State of Charge (Start %)',
       'State of Charge (End %)', 'Distance Driven (since last charge) (km)',
       'Temperature (°C)', 'Vehicle Age (years)'],
      dtype='object')

In [18]:
X_train_io = X_train.copy() ## DIFF IN  OUTPUT CU INPUT DATA NOT SCALED ELSE-- IT IS CORRECT

for col in con_col:
  X_train_io = outlier_impute_iqr(X_train_io,col)

Column--> Battery Capacity (kWh)
-2.3241696227235886 2.228495421995734 -2.3241696227235886 2.228495421995734
Column--> Energy Consumed (kWh)
-3.3271059378496632 3.3446660603796365 -1.950026882211301 3.3446660603796365
Column--> Charging Duration (hours)
-3.2682970989850366 3.2263772904346064 -2.0676213924362337 3.2263772904346064
Column--> Charging Rate (kW)
-3.309043836822947 3.264113001800868 -1.7479735039452566 3.264113001800868
Column--> Charging Cost (USD)
-3.4218791643536455 3.4333135841573217 -1.9756361885336213 3.4333135841573217
Column--> State of Charge (Start %)
-3.4419401990972824 3.39951514063654 -1.9107112674418218 3.041918129848564
Column--> State of Charge (End %)
-3.1149311902721966 3.1267312251564077 -3.1149311902721966 3.1267312251564077
Column--> Distance Driven (since last charge) (km)
-3.4765358075685264 3.5057930797496843 -1.7746298502964073 2.8085617566921934
Column--> Temperature (°C)
-3.304466717677972 3.325483864301752 -1.744866625970943 3.325483864301752
Col

In [19]:
# poly,RBF,cosine
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(kernel='rbf',random_state=7)
kpca.fit(X_train_io)

kpca.eigenvalues_.shape

(790,)